In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from helpers import make_preprocessor
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingRegressor

In [ ]:
df = pd.read_csv("data/ames_housing.csv")
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_df = train_df.drop(index=[523, 1298])
X_train = train_df.drop(columns=['Id', 'GarageYrBlt', 'PoolArea', 'SalePrice'])
y_train = train_df['SalePrice']

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of y_train: {y_train.shape}")

Shape of X_train: (1166, 77)
Shape of y_train: (1166,)


In [3]:
pipe = Pipeline([('prep', make_preprocessor(X_train)),
                 ('model', GradientBoostingRegressor(random_state=42))])

param_grid = {
    'model__n_estimators': [100, 300],
    'model__learning_rate': [0.05, 0.1],
    'model__max_depth': [2, 3, 4],
}

In [4]:
search = GridSearchCV(pipe, param_grid, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
search.fit(X_train, y_train)

print(search.best_params_)
print(-search.best_score_)

{'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__n_estimators': 300}
15227.158654180856


In [5]:
param_grid_2 = {
    'model__n_estimators': [300, 600, 1000],
    'model__learning_rate': [0.02, 0.05],
    'model__max_depth': [3],
}

search_2 = GridSearchCV(pipe, param_grid_2, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
search_2.fit(X_train, y_train)

print(search_2.best_params_)
print(-search_2.best_score_)

{'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__n_estimators': 1000}
14999.744197596132


In [7]:
cv = pd.DataFrame(search_2.cv_results_)
cv = cv[['param_model__n_estimators', 'param_model__learning_rate', 'mean_test_score', 'std_test_score']]
cv['MAE'] = -cv['mean_test_score']
cv.sort_values('MAE')[['param_model__n_estimators', 'param_model__learning_rate', 'MAE', 'std_test_score']].round(1)
view = cv.sort_values('MAE')[['param_model__n_estimators', 'param_model__learning_rate', 'MAE', 'std_test_score']]
view.round({'MAE': 1, 'std_test_score': 1})

,param_model__n_estimators,param_model__learning_rate,MAE,std_test_score
5,1000,0.05,14999.7,635.7
4,600,0.05,15019.8,722.3
2,1000,0.02,15087.6,993.1
3,300,0.05,15227.2,780.5
1,600,0.02,15389.6,1065.9
0,300,0.02,16288.3,1406.1


## Gradient Boosting: tuning result

Grid search with 5-fold cross-validation on the cleaned training set, scored by MAE.

| Setting | MAE |
|---|---|
| Untuned (defaults: 100 trees, learning rate 0.1, depth 3) | $15,863 |
| Search 1 best (300 trees, learning rate 0.05, depth 3) | $15,227 |
| Search 2 best (1,000 trees, learning rate 0.05, depth 3) | $15,000 |

The top three settings of the second search are within about $90 of each other, while the fold-to-fold spread is $640 to $990, so they are effectively tied. Going from 300 to 1,000 trees buys about $230 (less than one spread) at roughly three times the training time.

**Final choice: 300 trees, learning rate 0.05, depth 3 (MAE $15,227).** It is statistically tied with the best setting, faster and simpler. Tuning improved the MAE by about 4%.

These scores are slightly optimistic, because the best setting was chosen on the same folds it is scored on. The test set is the honest check, and it is only used once, at the end.